<a href="https://colab.research.google.com/github/ghalde194/APP1/blob/main/Erros_Cargos_Diferentes_o_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [50]:
import unicodedata
from google.colab import files
import pandas as pd
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter


def padronizar_texto(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).strip().upper()
    texto = (
        unicodedata.normalize("NFKD", texto)
        .encode("ascii", "ignore")
        .decode("utf-8")
    )
    return " ".join(texto.split())


def padronizar_matricula(valor):
    if pd.isna(valor):
        return ""
    try:
        return str(int(float(valor))).strip()
    except ValueError:
        return str(valor).strip()


# 1. PASSO: Selecionar o arquivo
print("Por favor, selecione o arquivo Excel com as duas abas...")
uploaded = files.upload()
nome_arquivo_origem = list(uploaded.keys())[0]

# 2. PASSO: Carregar os dados
print("\nLendo as abas do arquivo...")
xls = pd.ExcelFile(nome_arquivo_origem)
sheet_names = xls.sheet_names

datasul_sheet_name = next(
    (s for s in sheet_names if "datasul" in s.strip().lower()), None
)
workday_sheet_name = next(
    (s for s in sheet_names if "workday" in s.strip().lower()), None
)

df_datasul = pd.read_excel(nome_arquivo_origem, sheet_name=datasul_sheet_name)
df_workday = pd.read_excel(nome_arquivo_origem, sheet_name=workday_sheet_name)

df_datasul["Matrícula"] = df_datasul["Matrícula"].apply(padronizar_matricula)
df_workday["ID DATASUL"] = df_workday["ID DATASUL"].apply(padronizar_matricula)

# 3. PASSO: Selecionando apenas as colunas de origem necessárias
df_datasul_filtrado = df_datasul[
    ["Matrícula", "Nome", "Cargo Básico-Descrição"]
].copy()
df_workday_filtrado = df_workday[
    [
        "ID DATASUL",
        "Legal Name in General Display Format",
        "Business Title",
    ]
].copy()

df_workday_filtrado.rename(columns={"ID DATASUL": "Matrícula"}, inplace=True)

# 4. PASSO: Criando colunas auxiliares para comparação justa
df_datasul_filtrado["_cargo_comp"] = df_datasul_filtrado[
    "Cargo Básico-Descrição"
].apply(padronizar_texto)
df_workday_filtrado["_cargo_comp"] = df_workday_filtrado[
    "Business Title"
].apply(padronizar_texto)

# 5. PASSO: Cruzando os dados
df_comparacao = pd.merge(
    df_datasul_filtrado,
    df_workday_filtrado,
    on="Matrícula",
    how="inner",
    suffixes=("_DS", "_WD"),
)

# Funções de detecção de abreviação
termos_abreviados = [
    ("PLENO", "PL"),
    ("SENIOR", "SR"),
    ("JUNIOR", "JR"),
    ("COMERCIAL", "COM"),
    ("ADMINISTRACAO", "ADM"),
]


def eh_abreviacao(row):
    cargo_ds = row["_cargo_comp_DS"]
    cargo_wd = row["_cargo_comp_WD"]
    if cargo_ds == cargo_wd:
        return False
    for termo_longo, termo_curto in termos_abreviados:
        if (
            cargo_ds.replace(termo_longo, termo_curto) == cargo_wd
            or cargo_wd.replace(termo_longo, termo_curto) == cargo_ds
        ):
            return True
    palavras_ds = cargo_ds.split()
    palavras_wd = cargo_wd.split()
    if len(palavras_ds) == len(palavras_wd) and len(palavras_ds) > 0:
        for p_ds, p_wd in zip(palavras_ds, palavras_wd):
            if p_ds != p_wd and not (
                p_ds.startswith(p_wd) or p_wd.startswith(p_ds)
            ):
                return False
        return True
    return False


# AQUI ESTÁ O FILTRO PARA CARGOS DIFERENTES
# 1. Pegamos quem tem cargo diferente
df_diferentes = df_comparacao[
    df_comparacao["_cargo_comp_DS"] != df_comparacao["_cargo_comp_WD"]
].copy()

# 2. Excluímos as linhas que são apenas abreviações
mask_abreviacao = df_diferentes.apply(eh_abreviacao, axis=1)
df_cargos_realmente_diferentes = df_diferentes[~mask_abreviacao].copy()

# Organizando e Ordenando por Nome
df_cargos_realmente_diferentes = df_cargos_realmente_diferentes[
    [
        "Matrícula",
        "Nome",
        "Legal Name in General Display Format",
        "Cargo Básico-Descrição",
        "Business Title",
    ]
]
df_cargos_realmente_diferentes.columns = [
    "Matrícula",
    "Nome",
    "Nome (Workday)",
    "Cargo (Datasul)",
    "Cargo (Workday)",
]
df_cargos_realmente_diferentes = df_cargos_realmente_diferentes.sort_values(
    by="Nome", ascending=True
)

# Salvando no Excel e formatando a saída
nome_arquivo_saida = "Erros_Cargos_Diferentes.xlsx"


def formatar_aba(ws, df):
    header_fill = PatternFill(
        start_color="1F497D", end_color="1F497D", fill_type="solid"
    )
    error_fill = PatternFill(
        start_color="FFC7CE", end_color="FFC7CE", fill_type="solid"
    )

    thin_border = Border(
        left=Side(style="thin", color="D3D3D3"),
        right=Side(style="thin", color="D3D3D3"),
        top=Side(style="thin", color="D3D3D3"),
        bottom=Side(style="thin", color="D3D3D3"),
    )

    for col_num, header in enumerate(df.columns, 1):
        cell = ws.cell(row=1, column=col_num)
        cell.font = Font(color="FFFFFF", bold=True, name="Calibri", size=11)
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = thin_border

    for row in range(2, ws.max_row + 1):
        for col in range(1, ws.max_column + 1):
            cell = ws.cell(row=row, column=col)
            cell.border = thin_border
            cell.font = Font(name="Calibri", size=11)
            cell.alignment = Alignment(vertical="center")

            if col == 1:
                cell.alignment = Alignment(
                    horizontal="center", vertical="center"
                )

            # Pinta as colunas de cargo de vermelho claro para destacar o erro
            if col in [4, 5]:
                cell.fill = error_fill

    for col in ws.columns:
        max_len = max(len(str(cell.value or "")) for cell in col)
        col_letter = get_column_letter(col[0].column)
        ws.column_dimensions[col_letter].width = max(max_len + 3, 12)

    ws.freeze_panes = "A2"


with pd.ExcelWriter(nome_arquivo_saida, engine="openpyxl") as writer:
    df_cargos_realmente_diferentes.to_excel(
        writer, sheet_name="Cargos Diferentes", index=False
    )
    formatar_aba(writer.sheets["Cargos Diferentes"], df_cargos_realmente_diferentes)

print(f"\nProcesso concluído! Baixando {nome_arquivo_saida}...")
files.download(nome_arquivo_saida)

Por favor, selecione o arquivo Excel com as duas abas...


Saving Relatórios Funcionários 300326 - Leonardo.xlsx to Relatórios Funcionários 300326 - Leonardo (43).xlsx

Lendo as abas do arquivo...

Processo concluído! Baixando Erros_Cargos_Diferentes.xlsx...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>